# Group 3

In [1]:
!pip install ftfy regex tqdm torchmetrics git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-_9s5uj0a
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-_9s5uj0a
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 32.7 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=c5239fce06f287878830d2c8688edc732ecb4017585292cfc0c39d92f71c7bd0
  Stored in directory: /tmp/pip-ephem-wheel-cache-mf1a0739/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [8]:
import os

# Define paths
zip_path = './aml-group-lab.zip'
extract_to = './AMLdataset'

# Automated Unzip
if not os.path.exists(extract_to):
    os.makedirs(extract_to)
    print(f"Unzipping {zip_path}...")

    # Using the shell command is usually faster for large zips
    !unzip -q {zip_path} -d {extract_to}

    print("Unzip complete!")
else:
    print("Dataset already unzipped.")

Unzipping ./aml-group-lab.zip...
Unzip complete!


In [15]:
import os
import pandas as pd

from PIL import Image
from torch.utils.data import Dataset


class AMLDataset(Dataset):
    def __init__(self, train_csv_path, imgs_dir, df=None, train=True, transform=None):
        self.df = pd.read_csv(train_csv_path) if df is None else df
        self.imgs_dir = imgs_dir
        self.train = train
        self.transform = transform

        if self.train:
            # Create the 'classes' attribute (Unique list of names)
            # We sort them to ensure the index mapping is always consistent
            self.classes = sorted(self.df['label'].unique().tolist())

            # Create a mapping from Name -> Integer ID
            self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

            # Create the 'labels' attribute (The ID for every single row)
            # This is helpful if you want to use a Weighted or Balanced Sampler later
            self.labels = [self.class_to_idx[name] for name in self.df['label']]
        else:
            # Group the rows by episode_id so __len__ returns total number of tasks
            self.episode_ids = sorted(self.df['episode_id'].unique())

    def __len__(self):
        if self.train:
            return len(self.df)
        else:
            return len(self.episode_ids)

    def load_img(self, filename):
        img_path = os.path.join(self.imgs_dir, str(filename))
        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)
        return img

    def __getitem__(self, idx):
        if self.train:
            # Standard single image return
            row = self.df.iloc[idx]
            return self.load_img(row["filename"]), self.labels[idx]

        else:
            # Return a full 5-way 5-shot Episode
            ep_id = self.episode_ids[idx]
            ep_df = self.df[self.df['episode_id'] == ep_id]

            support_df = ep_df[ep_df['role'] == 'support']
            query_df = ep_df[ep_df['role'] == 'query']

            # Pack support images and their labels
            s_imgs = torch.stack([self.load_img(f) for f in support_df['filename']])
            s_labels = torch.tensor(support_df['label'].values)

            # Pack query images
            q_imgs = torch.stack([self.load_img(f) for f in query_df['filename']])

            # If your test CSV has query labels (sometimes they are hidden), include them:
            # q_labels = torch.tensor(query_df['label'].values) if 'label' in query_df.columns else None

            return {
                "support_imgs": s_imgs,
                "support_labels": s_labels,
                "query_imgs": q_imgs,
                "episode_id": ep_id
            }

In [16]:
import os
import clip
import torch
import torchmetrics
import pandas as pd

from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10

# Define constants
LR = 1e-6 # Learning Rate
BATCH_SIZE = 32 # Number of images processed in one iteration
SHOTS = 5 # 5-way 5-shot
EPOCHS = 5 # Number of epochs

# Define controls
best_acc = 0.0 # Used to save best checkpoints

# Load the models
device = "cuda" if torch.cuda.is_available() else "cpu"
student, preprocess = clip.load('ViT-B/32', device, jit=False)
teacher, _ = clip.load('ViT-B/32', device)
for param in teacher.parameters():
    param.requires_grad = False

# Download the dataset
# Load the full training CSV
full_df = pd.read_csv("./AMLdataset/release/train.csv")

# 2. Split into Train and Test (Stratified by label)
train_df, test_df = train_test_split(
    full_df,
    test_size=0.20,            # 20% for validation
    random_state=42,
    stratify=full_df['label']  # Ensures all 250 classes are in both
)

train_dataset = AMLDataset(
    train_csv_path="./AMLdataset/release/train.csv",
    df = train_df,
    imgs_dir="./AMLdataset/release/images/",
    train=True,
    transform=preprocess
)

test_dataset = AMLDataset(
    train_csv_path="./AMLdataset/release/train.csv",
    df = test_df,
    imgs_dir="./AMLdataset/release/images/",
    train=True,
    transform=preprocess
)

val_dataset = AMLDataset(
    train_csv_path="./AMLdataset/release/test_episodes_release.csv",
    imgs_dir="./AMLdataset/release/images/",
    train=False,
    transform=preprocess
)

# Prepare the data
train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,       # Standard way to randomize
    drop_last=True      # Helpful for CLIP to keep matrix dimensions consistent
)
num_batches_train = len(train_dataloader)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,       # Standard way to randomize
    drop_last=True      # Helpful for CLIP to keep matrix dimensions consistent
)
num_batches_test = len(test_dataloader)

all_class_texts = torch.cat([clip.tokenize(f"a photo of a {c}") for c in train_dataset.classes]).to(device)
NUM_CLASSES = len(train_dataset.classes)

# Prepare criterion and optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    [p for p in student.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=0.1
)

# Prepare the training
for epoch in range(EPOCHS):
    print(f"Epoch: {epoch}")
    epoch_train_loss = 0
    student.train() # Set the model to training mode (enables dropout/batchnorm updates)

    print("Running Training...")
    for batch in tqdm(train_dataloader, total=num_batches_train):
        optimizer.zero_grad() # Clear previous gradients before starting a new optimization step

        # Unpack images and class indices
        # images shape: [Batch, 3, 224, 224]
        # labels shape: [Batch]
        images, class_ids = batch
        images = images.to(device)

        # Map class IDs to text descriptions
        # We clean class names (replace underscores with spaces) for better CLIP performance
        texts = [f"a photo of a {str(train_dataset.classes[i]).replace('_', ' ')}" for i in class_ids]
        texts = clip.tokenize(texts).to(device)

        # Forward pass
        # CLIP computes similarity between all images and all texts in the batch
        logits_per_image, logits_per_text = student(images, texts)

        # Define Ground Truth
        # Creates a diagonal target [0, 1, ..., N-1] where image i matches text i
        ground_truth = torch.arange(len(images), dtype=torch.long, device=device)

        # Compute Symmetric Loss
        loss_i = criterion(logits_per_image, ground_truth)
        loss_t = criterion(logits_per_text, ground_truth)
        total_loss = (loss_i + loss_t) / 2

        # Optimization
        total_loss.backward() # Perform backpropagation to calculate gradients
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0) # This helps stability
        optimizer.step() # Update model weights based on calculated gradients

        epoch_train_loss += total_loss

    print(f"Epoch: {epoch} | Training Loss: {epoch_train_loss:.4f}")

Epoch: 0
Running Training...


100%|██████████| 125/125 [00:26<00:00,  4.74it/s]


Epoch: 0 | Training Loss: nan
Epoch: 1
Running Training...


100%|██████████| 125/125 [00:26<00:00,  4.78it/s]


Epoch: 1 | Training Loss: nan
Epoch: 2
Running Training...


...

Epoch: 2 | Training Loss: nan
Epoch: 3
Running Training...


100%|██████████| 125/125 [00:25<00:00,  4.83it/s]


Epoch: 3 | Training Loss: nan
Epoch: 4
Running Training...


100%|██████████| 125/125 [00:26<00:00,  4.81it/s]


Epoch: 4 | Training Loss: nan


In [22]:
import os
import clip
import torch
from torchvision.datasets import CIFAR100

# Load the model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = student

# Download the dataset
cifar100 = AMLDataset(
    train_csv_path="./AMLdataset/release/train.csv",
    df = test_df,
    imgs_dir="./AMLdataset/release/images/",
    train=True,
    transform=preprocess
)

# Prepare the inputs
image, class_id = cifar100[50]
image_input = image.unsqueeze(0).to(device)
text_inputs = torch.cat(
    [clip.tokenize(f"a photo of a {c}") for c in cifar100.classes]
).to(device)

# Calculate features
with torch.no_grad():
    image_features = model.encode_image(image_input)
    text_features = model.encode_text(text_inputs)

# Pick the top 5 most similar labels for the image
image_features /= image_features.norm(dim=-1, keepdim=True)
text_features /= text_features.norm(dim=-1, keepdim=True)
similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
values, indices = similarity[0].topk(5)

# Print the result
print("\nTop predictions:\n")
for value, index in zip(values, indices):
    print(f"{str(cifar100.classes[index]):>16s}: {100 * value.item():.2f}%")



Top predictions:

               1: nan%
               0: nan%
               2: nan%
               5: nan%
               4: nan%


In [ ]:
val_dataloader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=True,       # Standard way to randomize
    drop_last=True      # Helpful for CLIP to keep matrix dimensions consistent
)
num_batches_val = len(val_dataloader)

def evaluate_episodic(model, dataloader, device):
    model.eval()
    all_episode_accs = []

    print("Running Episodic Evaluation...")
    with torch.no_grad():
        for episode in tqdm(dataloader):
            # Setup (batch_size is 1, so we squeeze)
            s_imgs = episode["support_imgs"].squeeze(0).to(device)   # [25, 3, 224, 224]
            s_labels = episode["support_labels"].squeeze(0).to(device)
            q_imgs = episode["query_imgs"].squeeze(0).to(device)     # [25, 3, 224, 224]

            # Get Features
            s_feats = model.encode_image(s_imgs)
            q_feats = model.encode_image(q_imgs)

            # Normalize
            s_feats /= s_feats.norm(dim=-1, keepdim=True)
            q_feats /= q_feats.norm(dim=-1, keepdim=True)

            # Create Class Prototypes (The "Average" for each of the 5 classes)
            unique_labels = torch.unique(s_labels) # These are your 5 classes for this episode
            prototypes = []
            for label in unique_labels:
                # Find all support images that belong to this specific label and average them
                class_prototype = s_feats[s_labels == label].mean(dim=0)
                prototypes.append(class_prototype / class_prototype.norm())

            prototypes = torch.stack(prototypes) # Shape: [5, 512]

            # Classify Queries against Prototypes
            # [25 queries, 512] @ [512, 5 prototypes] -> [25, 5] similarity matrix
            logits = 100.0 * q_feats @ prototypes.T
            preds = logits.argmax(dim=-1)

            # Accuracy Calculation
            # Note: Few-shot query labels are usually 0-4 (indices of the unique_labels)
            # If your dataset provides true labels for queries, use them here:
            if "query_labels" in episode:
                q_labels = episode["query_labels"].squeeze(0).to(device)
                # Map the true labels to 0-4 range to match our prototypes index
                label_to_idx = {val.item(): i for i, val in enumerate(unique_labels)}
                target_indices = torch.tensor([label_to_idx[l.item()] for l in q_labels], device=device)

                acc = (preds == target_indices).float().mean()
                all_episode_accs.append(acc)

        final_acc = torch.stack(all_episode_accs).mean()
        print(f"\nFinal Episodic Accuracy: {final_acc:.2%}")
        return final_acc


evaluate_episodic(student, val_dataloader, device)